# FastText Text Classification

In this notebook we will demonstrate using the fastText library to perform text classification on the sentiment and emotion dataset.

fastText is a library for learning of word embeddings and text classification created by Facebook's AI Research (FAIR) lab. The model allows to create an unsupervised learning or supervised learning algorithm for obtaining vector representations for words.

**Key advantages:**
- Very fast training and prediction
- Good performance even with small datasets
- Handles out-of-vocabulary words well

In [17]:
# Install requirements
!pip install -q pandas fasttext


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [18]:
# Necessary imports
import os
import pandas as pd
import fasttext

In [19]:
# Load the dataset
filepath = "Data/Sentiment and Emotion in Text/train_data.csv"
df = pd.read_csv(filepath)

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (30000, 2)


,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [20]:
# Check class distribution
df['sentiment'].value_counts()

sentiment
worry         7433
neutral       6340
sadness       4828
happiness     2986
love          2068
surprise      1613
hate          1187
fun           1088
relief        1021
empty          659
enthusiasm     522
boredom        157
anger           98
Name: count, dtype: int64

In [21]:
# Let us take the top 3 categories and leave out the rest
shortlist = ['neutral', 'happiness', 'worry']
df_subset = df[df['sentiment'].isin(shortlist)]
print(f"Subset shape: {df_subset.shape}")

df_subset['sentiment'].value_counts()

Subset shape: (16759, 2)


sentiment
worry        7433
neutral      6340
happiness    2986
Name: count, dtype: int64

## Text Preprocessing

In [22]:
# Let's do some cleaning of this text
import unicodedata

def clean_it(text, normalize=True):
    # Replacing possible issues with data
    s = str(text).replace(',', ' ').replace('"', '').replace('\'', ' \' ').replace('.', ' . ').replace('(', ' ( ').\
            replace(')', ' ) ').replace('!', ' ! ').replace('?', ' ? ').replace(':', ' ').replace(';', ' ').lower()
    
    # Normalizing / encoding the text
    if normalize:
        s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8')
    
    return s

# Now let's define a function to prepare data for FastText
def clean_df(data, cleanit=False, shuffleit=False, encodeit=False, label_prefix='__label__'):
    # Create a copy
    df = data[['content', 'sentiment']].copy(deep=True)
    
    # Clean the text
    if cleanit:
        df['content'] = df['content'].apply(lambda x: clean_it(x, encodeit))
    
    # Format for FastText: __label__<class> <text>
    df['fasttext_format'] = label_prefix + df['sentiment'].astype(str) + ' ' + df['content']
    
    # Shuffle if requested
    if shuffleit:
        df = df.sample(frac=1).reset_index(drop=True)
    
    return df

In [23]:
%%time
# Transform the dataset using the clean function
df_cleaned = clean_df(df_subset, cleanit=True, shuffleit=True)
print(f"Cleaned data shape: {df_cleaned.shape}")

# Show sample
print("\nSample FastText formatted data:")
print(df_cleaned['fasttext_format'].head(3).values)

Cleaned data shape: (16759, 3)

Sample FastText formatted data:
['__label__worry yup  def swine flu .  i think it was the sausage . '
 '__label__neutral @ambbboo playing with lighters fire matches and grass'
 '__label__worry @kel_kel_17 ehh carnt stand hot weather']
CPU times: user 18.4 ms, sys: 2.58 ms, total: 21 ms
Wall time: 20.2 ms


In [24]:
# Split into train and test (80/20 split)
train_size = int(0.8 * len(df_cleaned))
df_train = df_cleaned.iloc[:train_size]
df_test = df_cleaned.iloc[train_size:]

print(f"Train size: {len(df_train)}")
print(f"Test size: {len(df_test)}")

Train size: 13407
Test size: 3352


In [25]:
# Write files to disk as fastText classifier API reads files from disk
os.makedirs('Data/fasttext_data', exist_ok=True)

train_file = 'Data/fasttext_data/train.txt'
test_file = 'Data/fasttext_data/test.txt'

df_train['fasttext_format'].to_csv(train_file, index=False, header=False)
df_test['fasttext_format'].to_csv(test_file, index=False, header=False)

print(f"Training data saved to: {train_file}")
print(f"Test data saved to: {test_file}")

Training data saved to: Data/fasttext_data/train.txt
Test data saved to: Data/fasttext_data/test.txt


Now that we have the train and test files written to disk in a format fastText wants, we are ready to use it for text classification!

In [26]:
%%time
# Using fastText for feature extraction and training
print("Training FastText model...\n")

model = fasttext.train_supervised(
    input=train_file,
    lr=1.0,              # Learning rate
    epoch=25,            # Number of epochs  
    wordNgrams=2,        # Use word bigrams
    dim=100,             # Embedding dimension
    loss='ova',          # One-vs-all loss for multi-class
    verbose=2            # Verbosity level
)

print("\nTraining complete!")

Training FastText model...



Read 0M words
Number of words:  23264
Number of labels: 3
Progress:  85.0% words/sec/thread: 1833481 lr:  0.150225 avg.loss:  0.392182 ETA:   0h 0m 0s


Training complete!
CPU times: user 3.9 s, sys: 276 ms, total: 4.17 s
Wall time: 911 ms


Progress: 100.0% words/sec/thread: 1631640 lr:  0.000000 avg.loss:  0.337830 ETA:   0h 0m 0s


## Model Evaluation

FastText provides Precision@k and Recall@k metrics:
- Precision@k: Proportion of correct labels among the k predicted labels
- Recall@k: Proportion of correct labels among the true labels that are in the top k predictions

In [27]:
# Test the model with different k values
print("="*80)
print("MODEL EVALUATION RESULTS")
print("="*80)

for k in range(1, 4):
    results = model.test(test_file, k=k)
    n_samples = results[0]
    precision = results[1]
    recall = results[2]
    
    print(f"Test Samples: {n_samples} | "
          f"Precision@{k}: {precision*100:2.4f}% | "
          f"Recall@{k}: {recall*100:2.4f}%")

print("="*80)

MODEL EVALUATION RESULTS
Test Samples: 3352 | Precision@1: 56.2649% | Recall@1: 56.2649%
Test Samples: 3352 | Precision@2: 42.6313% | Recall@2: 85.2625%
Test Samples: 3352 | Precision@3: 33.3333% | Recall@3: 100.0000%


In [28]:
# Test with custom examples
test_texts = [
    "I am so happy today!",
    "This is terrible news, I'm very concerned.",
    "The meeting is at 3pm.",
    "Everything is going great, feeling wonderful!",
    "I'm worried about the exam results."
]

print("\nTesting custom examples:")
print("="*80)

for text in test_texts:
    cleaned = clean_it(text)
    prediction = model.predict(cleaned)
    label = prediction[0][0].replace('__label__', '')
    confidence = prediction[1][0]
    
    print(f"Text: {text}")
    print(f"Prediction: {label} (confidence: {confidence:.4f})")
    print("-"*80)


Testing custom examples:
Text: I am so happy today!
Prediction: happiness (confidence: 1.0000)
--------------------------------------------------------------------------------
Text: This is terrible news, I'm very concerned.
Prediction: worry (confidence: 1.0000)
--------------------------------------------------------------------------------
Text: The meeting is at 3pm.
Prediction: neutral (confidence: 0.9526)
--------------------------------------------------------------------------------
Text: Everything is going great, feeling wonderful!
Prediction: happiness (confidence: 0.9992)
--------------------------------------------------------------------------------
Text: I'm worried about the exam results.
Prediction: worry (confidence: 1.0000)
--------------------------------------------------------------------------------


In [29]:
# Model information
print("\nModel Information:")
print("="*50)
print(f"Number of words: {len(model.words)}")
print(f"Labels: {model.labels}")
print(f"Dimension: {model.get_dimension()}")


Model Information:
Number of words: 23264
Labels: ['__label__worry', '__label__neutral', '__label__happiness']
Dimension: 100


## Summary

FastText achieved good results with minimal training time! The model is fast and efficient for text classification tasks.

**Key takeaways:**
- FastText is significantly faster than traditional ML classifiers (like LogisticRegression)
- Achieves competitive accuracy with simple architecture
- Works well even with relatively small datasets
- Handles word n-grams efficiently